In [1]:
import numpy as np
from sklearn.metrics import f1_score
import random
import time
import os
from joblib import Parallel, delayed

import src.utils as utils
from src.model import Nonneg_dyn_dag_learning
from src.baselines import Dynotears

SEED = 10
N_CPUS = os.cpu_count()

np.random.seed(SEED)
random.seed(SEED)

## Auxiliary functions

In [ ]:
def run_exps(g, data_p, exps, thr=.2, verb=False):

    W_true, _, A_true, _, X, Y = utils.simulate_svar(**data_p)
    W_true_bin = utils.to_bin(W_true, thr)
    A_true_bin = utils.to_bin(A_true, thr)

    T, N = X.shape
    n_lags = Y.shape[1] // N

    shd, fs_W, err_W, fs_A, err_A, acyc, runtime = [np.zeros(len(exps))  for _ in range(7)]
    for i, exp in enumerate(exps):
        args = exp['args'].copy()
        if 'adapt_lamb' in exp.keys() and exp['adapt_lamb']:
            args['lamb_W'] = utils.get_lamb_value(N, T, args['lamb_W'])
            args['lamb_A'] = utils.get_lamb_value(N*n_lags, T, args['lamb_A'])


        model = exp['model'](**exp['init']) if 'init' in exp.keys() else exp['model']()
        t_i = time.time()
        model.fit(X, Y, **args)
        t_solved = time.time() - t_i

        W_est = model.W_est
        A_est = model.A_est

        W_est_bin = utils.to_bin(W_est, thr)
        A_est_bin = utils.to_bin(A_est, thr)

        shd[i], _, _ = utils.count_accuracy(W_true_bin, W_est_bin)
        fs_W[i] = f1_score(W_true_bin.flatten(), W_est_bin.flatten())
        fs_A[i] = f1_score(A_true_bin.flatten(), A_est_bin.flatten())
        err_W[i] = utils.compute_norm_sq_err(W_true, W_est)
        err_A[i] = utils.compute_norm_sq_err(A_true, A_est)
        acyc[i] = model.dagness(W_est) if hasattr(model, 'dagness') else 1
        runtime[i] = t_solved

        if verb and (g % N_CPUS == 0):
            print(
                f'\t- {g+1}: {exp["leg"]} | '
                f'SHD={shd[i]}  |  F1(W)={fs_W[i]:.3f}  |  F1(A)={fs_A[i]:.3f}  |  '
                f'||W-Ŵ||²={err_W[i]:.3f}  |  ||A-Â||²={err_A[i]:.3f}  |  '
                f'dagness={acyc[i]:.3f}  |  time={runtime[i]:.3f}s'
            )

    return shd, fs_W, err_W, fs_A, err_A, acyc, runtime

## Tested Models

In [ ]:
Exps = [
    {'model': Nonneg_dyn_dag_learning, 'init': {'acyclicity': 'logdet', 'primal_opt': 'pgd'},
     'args': {'stepsize': 5e-4, 'alpha_0': 1, 'rho_0': .05, 's': 1, 'lamb_W': .01, 'lamb_A': .05,
              'iters_in': 10000, 'iters_out': 20, 'beta': 5}, 
     'adapt_lamb': True, 'leg': 'PGD'},

    # {'model': Nonneg_dyn_dag_learning, 'init': {'acyclicity': 'logdet', 'primal_opt': 'fista', 'restart': False},
    #  'args': {'stepsize': 5e-4, 'alpha_0': 1, 'rho_0': .05, 's': 1, 'lamb_W': .01, 'lamb_A': .05,
    #           'iters_in': 10000, 'iters_out': 20, 'beta': 5}, 
    #  'adapt_lamb': True, 'leg': 'APGD'},

    {'model': Nonneg_dyn_dag_learning, 'init': {'acyclicity': 'logdet', 'primal_opt': 'fista', 'restart': True},
     'args': {'stepsize': 5e-4, 'alpha_0': 1, 'rho_0': .05, 's': 1, 'lamb_W': .01, 'lamb_A': .05,
              'iters_in': 10000, 'iters_out': 20, 'beta': 5}, 
     'adapt_lamb': True, 'leg': 'APGD-r'},

    # {'model': Nonneg_dyn_dag_learning, 'init': {'acyclicity': 'logdet', 'primal_opt': 'sca'},
    #  'args': {'stepsize': 5e-4, 'alpha_0': 1, 'rho_0': .05, 's': 1, 'lamb_W': .01, 'lamb_A': .05,
    #           'iters_in': 10000, 'iters_out': 20, 'beta': 5}, 
    #  'adapt_lamb': True, 'leg': 'SCA'},

    {'model': Nonneg_dyn_dag_learning, 'init': {'acyclicity': 'logdet', 'primal_opt': 'pgd'},
     'args': {'stepsize': 5e-4, 'alpha_0': 1, 'rho_0': .05, 's': 1, 'lamb_W': .01, 'lamb_A': .05,
              'iters_in': 10000, 'iters_out': 20, 'beta': 5}, 
     'adapt_lamb': True, 'leg': 'PGD-fix'},

    {'model': Nonneg_dyn_dag_learning, 'init': {'acyclicity': 'logdet', 'primal_opt': 'fista', 'restart': True},
     'args': {'stepsize': 5e-4, 'alpha_0': 1, 'rho_0': .05, 's': 1, 'lamb_W': .01, 'lamb_A': .05,
              'iters_in': 10000, 'iters_out': 20, 'beta': 5}, 
     'adapt_lamb': True, 'leg': 'APGD-r-fix'},
    
    {'model': Nonneg_dyn_dag_learning, 'init': {'acyclicity': 'logdet', 'primal_opt': 'sca-adam'},
     'args': {'stepsize': 5e-4, 'alpha_0': 1, 'rho_0': .05, 's': 1, 'lamb_W': .01, 'lamb_A': .05,
              'iters_in': 10000, 'iters_out': 20, 'beta': 5}, 
     'adapt_lamb': True, 'leg': 'SCA-ADAM'},

     # Baselines
     {'model': Dynotears, 'init': {}, 'args': {},  'leg': 'DYNOTEARS'},
]

## Experiments - T=500

### Test th=.1

In [ ]:
verb = True
n_dags = 25
thr = .075
N = 50
data_params = {
    'n_nodes': N,
    'n_samples': 500, # 1000,
    'dag_graph_type': 'er',
    'dag_edges': 4*N,
    'dag_w_range': (.1, .5),
    'n_lags': 2,
    'lag_graph_type': 'er',
    'er_edges': N,
    'lag_w_range': (.1, .4),
    'exp_decay': 1.5,
    'noise_type': 'normal',
    'var': 1
}
print('CPUs employed:', N_CPUS)

t_init = time.time()
results = Parallel(n_jobs=N_CPUS)(delayed(run_exps)(g, data_params, Exps, thr, verb=verb) for g in range(n_dags))
ellapsed_time = (time.time() - t_init)/60
print(f'----- Solved in {ellapsed_time:.3f} minutes -----')

# Extract results
shd, fs_W, err_W, fs_A, err_A, acyc, runtime = zip(*results)
metrics = {'shd': shd, 'fs_W': fs_W, 'err_W': err_W, 'fs_A': fs_A, 'err_A': err_A, 'acyc': acyc, 'time': runtime}

CPUs employed: 168
	-1: PGD | SHD=31.0  |  F1(W)=0.909  |  F1(A)=0.763  |  ||W-Ŵ||²=0.077  |  ||A-Â||²=0.366  |  dagness=0.000  |  time=14.060s
	-1: APGD | SHD=32.0  |  F1(W)=0.901  |  F1(A)=0.768  |  ||W-Ŵ||²=0.095  |  ||A-Â||²=0.376  |  dagness=0.000  |  time=11.024s
	-1: APGD-r | SHD=32.0  |  F1(W)=0.901  |  F1(A)=0.768  |  ||W-Ŵ||²=0.095  |  ||A-Â||²=0.376  |  dagness=0.000  |  time=10.477s
	-1: SCA | SHD=187.0  |  F1(W)=0.323  |  F1(A)=0.198  |  ||W-Ŵ||²=1.270  |  ||A-Â||²=1.161  |  dagness=0.000  |  time=5.572s
	-1: SCA-ADAM | SHD=187.0  |  F1(W)=0.323  |  F1(A)=0.198  |  ||W-Ŵ||²=1.270  |  ||A-Â||²=1.161  |  dagness=0.000  |  time=0.363s
	-1: DYNOTEARS | SHD=56.0  |  F1(W)=0.808  |  F1(A)=0.723  |  ||W-Ŵ||²=0.204  |  ||A-Â||²=0.211  |  dagness=1.000  |  time=32.491s
----- Solved in 1.671 minutes -----


In [5]:
exps_leg = [exp['leg'] for exp in Exps]
utils.display_results(exps_leg, metrics, agg='mean')
utils.display_results(exps_leg, metrics, agg='median')

,leg,shd,fs_W,err_W,fs_A,err_A,acyc,time
0,PGD,39.7600 ± 37.3023,0.8715 ± 0.1523,0.1538 ± 0.3032,0.7436 ± 0.1603,0.5191 ± 0.3091,0.0000 ± 0.0000,13.2702 ± 4.3391
1,APGD,31.7200 ± 5.5894,0.9035 ± 0.0192,0.0904 ± 0.0199,0.7781 ± 0.0488,0.4549 ± 0.0704,0.0000 ± 0.0000,8.1042 ± 1.7424
2,APGD-r,31.9200 ± 5.7196,0.9030 ± 0.0195,0.0905 ± 0.0200,0.7773 ± 0.0482,0.4555 ± 0.0701,0.0000 ± 0.0000,6.9157 ± 1.6075
3,SCA,212.5200 ± 38.9657,0.3080 ± 0.1421,1.2226 ± 0.3605,0.1552 ± 0.0380,1.2717 ± 0.1021,0.0000 ± 0.0000,4.1944 ± 1.7114
4,SCA-ADAM,212.5200 ± 38.9534,0.3081 ± 0.1422,1.2226 ± 0.3605,0.1553 ± 0.0380,1.2718 ± 0.1022,-0.0000 ± 0.0000,1.6555 ± 3.0851
5,DYNOTEARS,53.6800 ± 8.2739,0.8229 ± 0.0312,0.1608 ± 0.0439,0.7672 ± 0.0623,0.2000 ± 0.0499,1.0000 ± 0.0000,31.3361 ± 7.1853


,leg,shd,fs_W,err_W,fs_A,err_A,acyc,time
0,PGD,33.0000 ± 37.3023,0.8997 ± 0.1523,0.0946 ± 0.3032,0.7765 ± 0.1603,0.4537 ± 0.3091,0.0000 ± 0.0000,12.2570 ± 4.3391
1,APGD,32.0000 ± 5.5894,0.9017 ± 0.0192,0.0926 ± 0.0199,0.7917 ± 0.0488,0.4476 ± 0.0704,0.0000 ± 0.0000,7.5542 ± 1.7424
2,APGD-r,32.0000 ± 5.7196,0.9017 ± 0.0195,0.0926 ± 0.0200,0.7917 ± 0.0482,0.4476 ± 0.0701,0.0000 ± 0.0000,6.5677 ± 1.6075
3,SCA,202.0000 ± 38.9657,0.3129 ± 0.1421,1.2527 ± 0.3605,0.1526 ± 0.0380,1.2842 ± 0.1021,0.0000 ± 0.0000,3.7949 ± 1.7114
4,SCA-ADAM,202.0000 ± 38.9534,0.3129 ± 0.1422,1.2534 ± 0.3605,0.1526 ± 0.0380,1.2842 ± 0.1022,0.0000 ± 0.0000,1.2146 ± 3.0851
5,DYNOTEARS,52.0000 ± 8.2739,0.8255 ± 0.0312,0.1567 ± 0.0439,0.7647 ± 0.0623,0.1905 ± 0.0499,1.0000 ± 0.0000,29.2995 ± 7.1853


## Experiments - T=1000

### Test th=.1

In [ ]:
verb = True
n_dags = 25
thr = .075
N = 50
data_params = {
    'n_nodes': N,
    'n_samples': 1000, # 1000,
    'dag_graph_type': 'er',
    'dag_edges': 4*N,
    'dag_w_range': (.1, .5),
    'n_lags': 2,
    'lag_graph_type': 'er',
    'er_edges': N,
    'lag_w_range': (.1, .4),
    'exp_decay': 1.5,
    'noise_type': 'normal',
    'var': 1
}
print('CPUs employed:', N_CPUS)

t_init = time.time()
results = Parallel(n_jobs=N_CPUS)(delayed(run_exps)(g, data_params, Exps, thr, verb=verb) for g in range(n_dags))
ellapsed_time = (time.time() - t_init)/60
print(f'----- Solved in {ellapsed_time:.3f} minutes -----')

# Extract results
shd, fs_W, err_W, fs_A, err_A, acyc, runtime = zip(*results)
metrics = {'shd': shd, 'fs_W': fs_W, 'err_W': err_W, 'fs_A': fs_A, 'err_A': err_A, 'acyc': acyc, 'time': runtime}

CPUs employed: 168
	-1: PGD | SHD=11.0  |  F1(W)=0.967  |  F1(A)=0.973  |  ||W-Ŵ||²=0.032  |  ||A-Â||²=0.244  |  dagness=0.000  |  time=10.939s
	-1: APGD | SHD=11.0  |  F1(W)=0.967  |  F1(A)=0.973  |  ||W-Ŵ||²=0.032  |  ||A-Â||²=0.244  |  dagness=0.000  |  time=5.341s
	-1: APGD-r | SHD=11.0  |  F1(W)=0.967  |  F1(A)=0.973  |  ||W-Ŵ||²=0.032  |  ||A-Â||²=0.244  |  dagness=0.000  |  time=4.253s
	-1: SCA | SHD=169.0  |  F1(W)=0.476  |  F1(A)=0.336  |  ||W-Ŵ||²=0.883  |  ||A-Â||²=1.058  |  dagness=0.000  |  time=3.983s
	-1: SCA-ADAM | SHD=169.0  |  F1(W)=0.476  |  F1(A)=0.336  |  ||W-Ŵ||²=0.883  |  ||A-Â||²=1.058  |  dagness=0.000  |  time=1.973s
	-1: DYNOTEARS | SHD=46.0  |  F1(W)=0.837  |  F1(A)=0.879  |  ||W-Ŵ||²=0.153  |  ||A-Â||²=0.075  |  dagness=1.000  |  time=29.750s
----- Solved in 2.008 minutes -----


In [7]:
exps_leg = [exp['leg'] for exp in Exps]
utils.display_results(exps_leg, metrics, agg='mean')
utils.display_results(exps_leg, metrics, agg='median')

,leg,shd,fs_W,err_W,fs_A,err_A,acyc,time
0,PGD,21.3600 ± 31.6036,0.9211 ± 0.1587,0.0996 ± 0.2861,0.8900 ± 0.1647,0.3039 ± 0.3111,0.0000 ± 0.0000,12.2394 ± 4.3995
1,APGD,14.1600 ± 3.4256,0.9562 ± 0.0131,0.0382 ± 0.0075,0.9232 ± 0.0329,0.2377 ± 0.0344,0.0000 ± 0.0000,7.3442 ± 1.1723
2,APGD-r,14.3200 ± 3.5521,0.9556 ± 0.0136,0.0470 ± 0.0420,0.9232 ± 0.0329,0.2379 ± 0.0342,0.0000 ± 0.0000,6.1520 ± 1.1128
3,SCA,165.2400 ± 53.9646,0.4874 ± 0.1909,0.8593 ± 0.4185,0.4146 ± 0.1214,0.9494 ± 0.0944,0.0000 ± 0.0000,6.0109 ± 2.4957
4,SCA-ADAM,165.1200 ± 54.0128,0.4878 ± 0.1913,0.8586 ± 0.4192,0.4147 ± 0.1218,0.9493 ± 0.0944,-0.0000 ± 0.0000,4.6500 ± 7.0741
5,DYNOTEARS,46.3200 ± 7.6405,0.8495 ± 0.0235,0.1235 ± 0.0342,0.8309 ± 0.0448,0.0871 ± 0.0164,1.0000 ± 0.0000,44.2592 ± 13.2411


,leg,shd,fs_W,err_W,fs_A,err_A,acyc,time
0,PGD,16.0000 ± 31.6036,0.9484 ± 0.1587,0.0400 ± 0.2861,0.9315 ± 0.1647,0.2385 ± 0.3111,0.0000 ± 0.0000,11.0838 ± 4.3995
1,APGD,14.0000 ± 3.4256,0.9582 ± 0.0131,0.0365 ± 0.0075,0.9315 ± 0.0329,0.2354 ± 0.0344,0.0000 ± 0.0000,7.1443 ± 1.1723
2,APGD-r,14.0000 ± 3.5521,0.9582 ± 0.0136,0.0370 ± 0.0420,0.9315 ± 0.0329,0.2354 ± 0.0342,0.0000 ± 0.0000,6.0807 ± 1.1128
3,SCA,187.0000 ± 53.9646,0.4650 ± 0.1909,0.8828 ± 0.4185,0.3598 ± 0.1214,0.9635 ± 0.0944,0.0000 ± 0.0000,5.1850 ± 2.4957
4,SCA-ADAM,187.0000 ± 54.0128,0.4637 ± 0.1913,0.8826 ± 0.4192,0.3598 ± 0.1218,0.9636 ± 0.0944,0.0000 ± 0.0000,1.3390 ± 7.0741
5,DYNOTEARS,46.0000 ± 7.6405,0.8492 ± 0.0235,0.1206 ± 0.0342,0.8471 ± 0.0448,0.0834 ± 0.0164,1.0000 ± 0.0000,41.0595 ± 13.2411


## Experiments - T = 5000

### Test th=.1

In [ ]:
verb = True
n_dags = 25
thr = .075
N = 50
data_params = {
    'n_nodes': N,
    'n_samples': 5000,
    'dag_graph_type': 'er',
    'dag_edges': 4*N,
    'dag_w_range': (.1, .5),
    'n_lags': 2,
    'lag_graph_type': 'er',
    'er_edges': N,
    'lag_w_range': (.1, .4),
    'exp_decay': 1.5,
    'noise_type': 'normal',
    'var': 1
}
print('CPUs employed:', N_CPUS)

t_init = time.time()
results = Parallel(n_jobs=N_CPUS)(delayed(run_exps)(g, data_params, Exps, thr, verb=verb) for g in range(n_dags))
ellapsed_time = (time.time() - t_init)/60
print(f'----- Solved in {ellapsed_time:.3f} minutes -----')

# Extract results
shd, fs_W, err_W, fs_A, err_A, acyc, runtime = zip(*results)
metrics = {'shd': shd, 'fs_W': fs_W, 'err_W': err_W, 'fs_A': fs_A, 'err_A': err_A, 'acyc': acyc, 'time': runtime}

CPUs employed: 168
	-1: PGD | SHD=4.0  |  F1(W)=0.989  |  F1(A)=1.000  |  ||W-Ŵ||²=0.005  |  ||A-Â||²=0.046  |  dagness=0.000  |  time=11.097s
	-1: APGD | SHD=4.0  |  F1(W)=0.989  |  F1(A)=1.000  |  ||W-Ŵ||²=0.005  |  ||A-Â||²=0.046  |  dagness=0.000  |  time=4.483s
	-1: APGD-r | SHD=4.0  |  F1(W)=0.989  |  F1(A)=1.000  |  ||W-Ŵ||²=0.005  |  ||A-Â||²=0.046  |  dagness=0.000  |  time=4.496s
	-1: SCA | SHD=180.0  |  F1(W)=0.473  |  F1(A)=0.588  |  ||W-Ŵ||²=0.950  |  ||A-Â||²=0.613  |  dagness=0.000  |  time=2.703s
	-1: SCA-ADAM | SHD=180.0  |  F1(W)=0.473  |  F1(A)=0.588  |  ||W-Ŵ||²=0.950  |  ||A-Â||²=0.613  |  dagness=0.000  |  time=1.125s
	-1: DYNOTEARS | SHD=42.0  |  F1(W)=0.851  |  F1(A)=0.841  |  ||W-Ŵ||²=0.093  |  ||A-Â||²=0.067  |  dagness=1.000  |  time=113.051s
----- Solved in 6.070 minutes -----


In [9]:
exps_leg = [exp['leg'] for exp in Exps]
utils.display_results(exps_leg, metrics, agg='mean')
utils.display_results(exps_leg, metrics, agg='median')

,leg,shd,fs_W,err_W,fs_A,err_A,acyc,time
0,PGD,26.4800 ± 57.3174,0.9160 ± 0.1922,0.1205 ± 0.3381,0.9033 ± 0.1802,0.1477 ± 0.3160,0.0051 ± 0.0249,12.1594 ± 6.8441
1,APGD,6.2800 ± 2.6611,0.9829 ± 0.0076,0.0067 ± 0.0025,0.9593 ± 0.0212,0.0488 ± 0.0095,0.0000 ± 0.0000,4.8605 ± 0.6197
2,APGD-r,7.1200 ± 3.4678,0.9807 ± 0.0093,0.0222 ± 0.0423,0.9593 ± 0.0212,0.0489 ± 0.0094,0.0000 ± 0.0000,4.6355 ± 0.9348
3,SCA,133.8400 ± 72.1336,0.6028 ± 0.2272,0.6287 ± 0.4716,0.7320 ± 0.1866,0.5156 ± 0.1822,0.0000 ± 0.0000,6.9391 ± 2.5609
4,SCA-ADAM,133.6000 ± 72.2518,0.6036 ± 0.2279,0.6282 ± 0.4717,0.7327 ± 0.1871,0.5149 ± 0.1828,0.0000 ± 0.0000,5.2649 ± 8.4620
5,DYNOTEARS,41.5600 ± 8.6814,0.8721 ± 0.0263,0.0844 ± 0.0345,0.8301 ± 0.0621,0.0605 ± 0.0139,1.0000 ± 0.0000,152.9502 ± 70.3374


,leg,shd,fs_W,err_W,fs_A,err_A,acyc,time
0,PGD,7.0000 ± 57.3174,0.9801 ± 0.1922,0.0092 ± 0.3381,0.9600 ± 0.1802,0.0517 ± 0.3160,0.0000 ± 0.0249,9.7992 ± 6.8441
1,APGD,6.0000 ± 2.6611,0.9838 ± 0.0076,0.0055 ± 0.0025,0.9620 ± 0.0212,0.0471 ± 0.0095,0.0000 ± 0.0000,4.8887 ± 0.6197
2,APGD-r,6.0000 ± 3.4678,0.9826 ± 0.0093,0.0066 ± 0.0423,0.9620 ± 0.0212,0.0471 ± 0.0094,0.0000 ± 0.0000,4.9228 ± 0.9348
3,SCA,137.0000 ± 72.1336,0.6682 ± 0.2272,0.4673 ± 0.4716,0.7736 ± 0.1866,0.4609 ± 0.1822,0.0000 ± 0.0000,7.4978 ± 2.5609
4,SCA-ADAM,137.0000 ± 72.2518,0.6682 ± 0.2279,0.4672 ± 0.4717,0.7736 ± 0.1871,0.4608 ± 0.1828,0.0000 ± 0.0000,1.7140 ± 8.4620
5,DYNOTEARS,41.0000 ± 8.6814,0.8747 ± 0.0263,0.0793 ± 0.0345,0.8312 ± 0.0621,0.0588 ± 0.0139,1.0000 ± 0.0000,125.2992 ± 70.3374


## Experiments - N=100, T=1000

### Test th=.1

In [13]:
verb = True
n_dags = 25
thr = .075
N = 100
data_params = {
    'n_nodes': N,
    'n_samples': 1000, # 1000,
    'dag_graph_type': 'er',
    'dag_edges': 4*N,
    'dag_w_range': (.1, .5),
    'n_lags': 2,
    'lag_graph_type': 'er',
    'er_edges': N,
    'lag_w_range': (.1, .4),
    'exp_decay': 1.5,
    'noise_type': 'normal',
    'var': 1
}
print('CPUs employed:', N_CPUS)

t_init = time.time()
results = Parallel(n_jobs=N_CPUS)(delayed(run_exps)(g, data_params, Exps, thr, verb=verb) for g in range(n_dags))
ellapsed_time = (time.time() - t_init)/60
print(f'----- Solved in {ellapsed_time:.3f} minutes -----')

# Extract results
shd, fs_W, err_W, fs_A, err_A, acyc, runtime = zip(*results)
metrics = {'shd': shd, 'fs_W': fs_W, 'err_W': err_W, 'fs_A': fs_A, 'err_A': err_A, 'acyc': acyc, 'time': runtime}

CPUs employed: 168
	-1: PGD | SHD=42.0  |  F1(W)=0.938  |  F1(A)=0.828  |  ||W-Ŵ||²=0.056  |  ||A-Â||²=0.413  |  dagness=0.000  |  time=61.930s
	-1: APGD | SHD=42.0  |  F1(W)=0.938  |  F1(A)=0.828  |  ||W-Ŵ||²=0.057  |  ||A-Â||²=0.413  |  dagness=0.000  |  time=39.403s
	-1: APGD-r | SHD=42.0  |  F1(W)=0.938  |  F1(A)=0.828  |  ||W-Ŵ||²=0.057  |  ||A-Â||²=0.413  |  dagness=0.000  |  time=28.410s
	-1: SCA | SHD=500.0  |  F1(W)=0.155  |  F1(A)=0.141  |  ||W-Ŵ||²=1.565  |  ||A-Â||²=1.364  |  dagness=0.000  |  time=12.241s
	-1: SCA-ADAM | SHD=500.0  |  F1(W)=0.155  |  F1(A)=0.141  |  ||W-Ŵ||²=1.565  |  ||A-Â||²=1.364  |  dagness=0.000  |  time=4.905s
	-1: DYNOTEARS | SHD=81.0  |  F1(W)=0.882  |  F1(A)=0.742  |  ||W-Ŵ||²=0.070  |  ||A-Â||²=0.089  |  dagness=1.000  |  time=173.581s
----- Solved in 7.809 minutes -----


In [14]:
exps_leg = [exp['leg'] for exp in Exps]
utils.display_results(exps_leg, metrics, agg='mean')
utils.display_results(exps_leg, metrics, agg='median')

,leg,shd,fs_W,err_W,fs_A,err_A,acyc,time
0,PGD,39.0000 ± 7.3485,0.9406 ± 0.0113,0.0560 ± 0.0065,0.9004 ± 0.0262,0.3747 ± 0.0425,0.0000 ± 0.0000,65.8121 ± 6.7739
1,APGD,172.5200 ± 313.1004,0.7568 ± 0.3679,0.4361 ± 0.7619,0.7940 ± 0.2776,0.5527 ± 0.4769,0.0000 ± 0.0000,45.4457 ± 9.4450
2,APGD-r,80.5200 ± 113.3671,0.8361 ± 0.2854,0.1959 ± 0.4229,0.8928 ± 0.0360,0.3847 ± 0.0632,0.0000 ± 0.0000,37.5852 ± 5.2529
3,SCA,487.3600 ± 97.6544,0.2911 ± 0.1371,1.2328 ± 0.3284,0.2090 ± 0.0495,1.2483 ± 0.0667,0.0000 ± 0.0000,24.4127 ± 10.3376
4,SCA-ADAM,487.5200 ± 97.4967,0.2909 ± 0.1369,1.2335 ± 0.3281,0.2091 ± 0.0496,1.2483 ± 0.0667,-0.0000 ± 0.0000,27.6014 ± 40.8571
5,DYNOTEARS,77.3200 ± 10.4679,0.8842 ± 0.0172,0.0628 ± 0.0100,0.8213 ± 0.0339,0.0838 ± 0.0107,1.0000 ± 0.0000,140.9782 ± 42.4488


,leg,shd,fs_W,err_W,fs_A,err_A,acyc,time
0,PGD,39.0000 ± 7.3485,0.9388 ± 0.0113,0.0557 ± 0.0065,0.9000 ± 0.0262,0.3612 ± 0.0425,0.0000 ± 0.0000,64.1593 ± 6.7739
1,APGD,42.0000 ± 313.1004,0.9339 ± 0.3679,0.0575 ± 0.7619,0.8927 ± 0.2776,0.4000 ± 0.4769,0.0000 ± 0.0000,44.8891 ± 9.4450
2,APGD-r,40.0000 ± 113.3671,0.9361 ± 0.2854,0.0575 ± 0.4229,0.8936 ± 0.0360,0.3603 ± 0.0632,0.0000 ± 0.0000,37.0146 ± 5.2529
3,SCA,492.0000 ± 97.6544,0.2504 ± 0.1371,1.3181 ± 0.3284,0.1995 ± 0.0495,1.2541 ± 0.0667,0.0000 ± 0.0000,20.0358 ± 10.3376
4,SCA-ADAM,493.0000 ± 97.4967,0.2504 ± 0.1369,1.3292 ± 0.3281,0.1998 ± 0.0496,1.2541 ± 0.0667,0.0000 ± 0.0000,5.9558 ± 40.8571
5,DYNOTEARS,76.0000 ± 10.4679,0.8880 ± 0.0172,0.0620 ± 0.0100,0.8219 ± 0.0339,0.0821 ± 0.0107,1.0000 ± 0.0000,127.3291 ± 42.4488


## Experiments - N=100 T = 5000

### Test th=.1

In [15]:
verb = True
n_dags = 25
thr = .075
N = 100
data_params = {
    'n_nodes': N,
    'n_samples': 5000,
    'dag_graph_type': 'er',
    'dag_edges': 4*N,
    'dag_w_range': (.1, .5),
    'n_lags': 2,
    'lag_graph_type': 'er',
    'er_edges': N,
    'lag_w_range': (.1, .4),
    'exp_decay': 1.5,
    'noise_type': 'normal',
    'var': 1
}
print('CPUs employed:', N_CPUS)

t_init = time.time()
results = Parallel(n_jobs=N_CPUS)(delayed(run_exps)(g, data_params, Exps, thr, verb=verb) for g in range(n_dags))
ellapsed_time = (time.time() - t_init)/60
print(f'----- Solved in {ellapsed_time:.3f} minutes -----')

# Extract results
shd, fs_W, err_W, fs_A, err_A, acyc, runtime = zip(*results)
metrics = {'shd': shd, 'fs_W': fs_W, 'err_W': err_W, 'fs_A': fs_A, 'err_A': err_A, 'acyc': acyc, 'time': runtime}

CPUs employed: 168
	-1: PGD | SHD=1.0  |  F1(W)=0.998  |  F1(A)=0.990  |  ||W-Ŵ||²=0.009  |  ||A-Â||²=0.062  |  dagness=0.000  |  time=50.074s
	-1: APGD | SHD=1450.0  |  F1(W)=0.078  |  F1(A)=0.913  |  ||W-Ŵ||²=1.918  |  ||A-Â||²=0.267  |  dagness=0.000  |  time=47.370s
	-1: APGD-r | SHD=416.0  |  F1(W)=0.058  |  F1(A)=0.990  |  ||W-Ŵ||²=1.067  |  ||A-Â||²=0.073  |  dagness=0.000  |  time=51.150s
	-1: SCA | SHD=197.0  |  F1(W)=0.734  |  F1(A)=0.819  |  ||W-Ŵ||²=0.289  |  ||A-Â||²=0.488  |  dagness=0.000  |  time=42.785s
	-1: SCA-ADAM | SHD=197.0  |  F1(W)=0.736  |  F1(A)=0.816  |  ||W-Ŵ||²=0.288  |  ||A-Â||²=0.488  |  dagness=0.000  |  time=52.419s
	-1: DYNOTEARS | SHD=72.0  |  F1(W)=0.899  |  F1(A)=0.940  |  ||W-Ŵ||²=0.042  |  ||A-Â||²=0.037  |  dagness=1.000  |  time=384.928s
----- Solved in 14.302 minutes -----


In [16]:
exps_leg = [exp['leg'] for exp in Exps]
utils.display_results(exps_leg, metrics, agg='mean')
utils.display_results(exps_leg, metrics, agg='median')

,leg,shd,fs_W,err_W,fs_A,err_A,acyc,time
0,PGD,3.7200 ± 1.9498,0.9924 ± 0.0040,0.0112 ± 0.0030,0.9784 ± 0.0102,0.0798 ± 0.0130,0.0000 ± 0.0000,51.0240 ± 4.9579
1,APGD,234.0400 ± 500.8114,0.8081 ± 0.3737,0.3935 ± 0.7690,0.8581 ± 0.2989,0.3092 ± 0.5592,0.0000 ± 0.0000,32.7505 ± 7.5591
2,APGD-r,20.0400 ± 80.9471,0.9559 ± 0.1835,0.0550 ± 0.2071,0.9786 ± 0.0101,0.0799 ± 0.0130,0.0000 ± 0.0000,29.9817 ± 7.8524
3,SCA,310.1600 ± 135.4786,0.5822 ± 0.1745,0.5978 ± 0.3610,0.6274 ± 0.1626,0.6412 ± 0.1245,0.0000 ± 0.0000,38.3996 ± 7.4730
4,SCA-ADAM,309.8000 ± 135.8143,0.5827 ± 0.1747,0.5976 ± 0.3622,0.6278 ± 0.1631,0.6409 ± 0.1248,0.0000 ± 0.0000,82.8286 ± 52.2241
5,DYNOTEARS,51.9600 ± 9.3187,0.9241 ± 0.0140,0.0473 ± 0.0125,0.8872 ± 0.0341,0.0465 ± 0.0082,1.0000 ± 0.0000,425.0839 ± 138.1648


,leg,shd,fs_W,err_W,fs_A,err_A,acyc,time
0,PGD,4.0000 ± 1.9498,0.9917 ± 0.0040,0.0103 ± 0.0030,0.9794 ± 0.0102,0.0794 ± 0.0130,0.0000 ± 0.0000,51.7406 ± 4.9579
1,APGD,3.0000 ± 500.8114,0.9938 ± 0.3737,0.0096 ± 0.7690,0.9756 ± 0.2989,0.0877 ± 0.5592,0.0000 ± 0.0000,31.6341 ± 7.5591
2,APGD-r,3.0000 ± 80.9471,0.9944 ± 0.1835,0.0094 ± 0.2071,0.9794 ± 0.0101,0.0793 ± 0.0130,0.0000 ± 0.0000,31.2017 ± 7.8524
3,SCA,291.0000 ± 135.4786,0.6192 ± 0.1745,0.4981 ± 0.3610,0.6022 ± 0.1626,0.6160 ± 0.1245,0.0000 ± 0.0000,39.4744 ± 7.4730
4,SCA-ADAM,291.0000 ± 135.8143,0.6192 ± 0.1747,0.5004 ± 0.3622,0.6045 ± 0.1631,0.6160 ± 0.1248,0.0000 ± 0.0000,111.0257 ± 52.2241
5,DYNOTEARS,50.0000 ± 9.3187,0.9265 ± 0.0140,0.0452 ± 0.0125,0.8901 ± 0.0341,0.0480 ± 0.0082,1.0000 ± 0.0000,391.0654 ± 138.1648


## Experiments - 3 lags

### Test th=.1

In [17]:
verb = True
n_dags = 25
thr = .075
N = 50
data_params = {
    'n_nodes': N,
    'n_samples': 5000,
    'dag_graph_type': 'er',
    'dag_edges': 4*N,
    'dag_w_range': (.1, .5),
    'n_lags': 2,
    'lag_graph_type': 'er',
    'er_edges': N,
    'lag_w_range': (.1, .4),
    'exp_decay': 1.5,
    'noise_type': 'normal',
    'var': 1
}
print('CPUs employed:', N_CPUS)

t_init = time.time()
results = Parallel(n_jobs=N_CPUS)(delayed(run_exps)(g, data_params, Exps, thr, verb=verb) for g in range(n_dags))
ellapsed_time = (time.time() - t_init)/60
print(f'----- Solved in {ellapsed_time:.3f} minutes -----')

# Extract results
shd, fs_W, err_W, fs_A, err_A, acyc, runtime = zip(*results)
metrics = {'shd': shd, 'fs_W': fs_W, 'err_W': err_W, 'fs_A': fs_A, 'err_A': err_A, 'acyc': acyc, 'time': runtime}

CPUs employed: 168
	-1: PGD | SHD=1.0  |  F1(W)=0.997  |  F1(A)=1.000  |  ||W-Ŵ||²=0.006  |  ||A-Â||²=0.051  |  dagness=0.000  |  time=9.729s
	-1: APGD | SHD=1.0  |  F1(W)=0.997  |  F1(A)=1.000  |  ||W-Ŵ||²=0.006  |  ||A-Â||²=0.051  |  dagness=0.000  |  time=4.937s
	-1: APGD-r | SHD=1.0  |  F1(W)=0.997  |  F1(A)=1.000  |  ||W-Ŵ||²=0.006  |  ||A-Â||²=0.051  |  dagness=0.000  |  time=4.943s
	-1: SCA | SHD=65.0  |  F1(W)=0.813  |  F1(A)=0.957  |  ||W-Ŵ||²=0.162  |  ||A-Â||²=0.284  |  dagness=0.000  |  time=8.190s
	-1: SCA-ADAM | SHD=65.0  |  F1(W)=0.813  |  F1(A)=0.957  |  ||W-Ŵ||²=0.156  |  ||A-Â||²=0.284  |  dagness=0.000  |  time=23.991s
	-1: DYNOTEARS | SHD=48.0  |  F1(W)=0.843  |  F1(A)=0.902  |  ||W-Ŵ||²=0.093  |  ||A-Â||²=0.058  |  dagness=1.000  |  time=106.493s
----- Solved in 3.937 minutes -----


In [18]:
exps_leg = [exp['leg'] for exp in Exps]
utils.display_results(exps_leg, metrics, agg='mean')
utils.display_results(exps_leg, metrics, agg='median')

,leg,shd,fs_W,err_W,fs_A,err_A,acyc,time
0,PGD,1.8400 ± 1.3470,0.9923 ± 0.0055,0.0083 ± 0.0030,0.9785 ± 0.0165,0.0474 ± 0.0098,0.0000 ± 0.0000,9.5053 ± 0.9741
1,APGD,1.3200 ± 0.9683,0.9947 ± 0.0040,0.0071 ± 0.0023,0.9785 ± 0.0165,0.0471 ± 0.0098,0.0000 ± 0.0000,5.1042 ± 0.8279
2,APGD-r,1.3200 ± 0.9683,0.9947 ± 0.0040,0.0071 ± 0.0023,0.9785 ± 0.0165,0.0471 ± 0.0098,0.0000 ± 0.0000,4.8210 ± 0.9718
3,SCA,107.3600 ± 78.1582,0.6855 ± 0.2144,0.4590 ± 0.4326,0.7113 ± 0.2152,0.4560 ± 0.1937,0.0000 ± 0.0000,7.5404 ± 2.3469
4,SCA-ADAM,107.2800 ± 78.4849,0.6857 ± 0.2152,0.4602 ± 0.4323,0.7115 ± 0.2158,0.4555 ± 0.1942,0.0000 ± 0.0000,11.0245 ± 10.9235
5,DYNOTEARS,34.5600 ± 10.1748,0.8893 ± 0.0311,0.0906 ± 0.0318,0.8752 ± 0.0315,0.0580 ± 0.0150,1.0000 ± 0.0000,123.7814 ± 40.2834


,leg,shd,fs_W,err_W,fs_A,err_A,acyc,time
0,PGD,1.0000 ± 1.3470,0.9944 ± 0.0055,0.0070 ± 0.0030,0.9778 ± 0.0165,0.0469 ± 0.0098,0.0000 ± 0.0000,9.7285 ± 0.9741
1,APGD,1.0000 ± 0.9683,0.9951 ± 0.0040,0.0060 ± 0.0023,0.9778 ± 0.0165,0.0457 ± 0.0098,0.0000 ± 0.0000,5.1072 ± 0.8279
2,APGD-r,1.0000 ± 0.9683,0.9951 ± 0.0040,0.0060 ± 0.0023,0.9778 ± 0.0165,0.0457 ± 0.0098,0.0000 ± 0.0000,4.9431 ± 0.9718
3,SCA,91.0000 ± 78.1582,0.7342 ± 0.2144,0.2836 ± 0.4326,0.7826 ± 0.2152,0.3898 ± 0.1937,0.0000 ± 0.0000,8.2792 ± 2.3469
4,SCA-ADAM,92.0000 ± 78.4849,0.7291 ± 0.2152,0.2953 ± 0.4323,0.7770 ± 0.2158,0.3880 ± 0.1942,0.0000 ± 0.0000,2.2933 ± 10.9235
5,DYNOTEARS,35.0000 ± 10.1748,0.8937 ± 0.0311,0.0771 ± 0.0318,0.8750 ± 0.0315,0.0583 ± 0.0150,1.0000 ± 0.0000,111.5612 ± 40.2834
